In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import os

from pathlib import Path
base_path = os.environ.get("TRAJRISK_ROOT") or str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "metadata" / "source_provenance.json").is_file()))
data_processed = os.path.join(base_path, "Data/processed")
results_figs   = os.path.join(base_path, "results/figures")

adata = sc.read_h5ad(os.path.join(data_processed, "adata_integrated.h5ad"))

pseudo_df = pd.read_csv(os.path.join(data_processed, "pseudotime_walesky.csv"), index_col=0)
pseudo_df['pseudotime'] = pd.to_numeric(pseudo_df['pseudotime'], errors='coerce')

common_cells = adata.obs_names.intersection(pseudo_df.index)
print(f"AnnData cells:      {adata.n_obs:,}")
print(f"Pseudotime cells:   {len(pseudo_df):,}")
print(f"Matched cells:      {len(common_cells):,}")

if len(common_cells) != adata.n_obs:
    print(f"⚠️  {adata.n_obs - len(common_cells):,} cells not matched — "
          f"will be NaN in pseudotime column")

adata.obs['pseudotime'] = np.nan
adata.obs.loc[common_cells, 'pseudotime'] = pseudo_df.loc[common_cells, 'pseudotime'].values

n_valid    = adata.obs['pseudotime'].notna().sum()
n_infinite = adata.obs['pseudotime'].isna().sum()

print(f"\nValid pseudotime values: {n_valid:,}")
print(f"NaN (Inf from R):        {n_infinite:,}")

# Validate correlation
valid_mask = (
    adata.obs['pseudotime'].notna() &
    adata.obs['timepoint_hours'].notna()
)

rho, pval = spearmanr(
    adata.obs.loc[valid_mask, 'pseudotime'],
    adata.obs.loc[valid_mask, 'timepoint_hours']
)

print(f"\nSpearman ρ (pseudotime vs timepoint_hours): {rho:.4f}")
print(f"p-value: {pval:.2e}")

# UMAP visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

sc.pl.umap(adata, color='pseudotime',
           ax=axes[0], show=False, frameon=False,
           color_map='plasma', title='Pseudotime')

sc.pl.umap(adata, color='timepoint_str',
           ax=axes[1], show=False, frameon=False,
           title='Timepoint')

sc.pl.umap(adata, color='treatment',
           ax=axes[2], show=False, frameon=False,
           title='Treatment')

plt.tight_layout()
plt.savefig(os.path.join(results_figs, "pseudotime_umap.png"), dpi=300, bbox_inches='tight')
plt.show()

output_path = os.path.join(data_processed, "adata_integrated_with_trajectory.h5ad")
adata.write_h5ad(output_path)

print(f"\n✅ Saved: {output_path}")
print(f"   Cells: {adata.n_obs:,}")
print(f"   Pseudotime range: "
      f"{adata.obs['pseudotime'].min():.3f} — "
      f"{adata.obs['pseudotime'].max():.3f}")

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from pathlib import Path
base_path = os.environ.get("TRAJRISK_ROOT") or str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "metadata" / "source_provenance.json").is_file()))
data_processed = os.path.join(base_path, "Data/processed")

adata = sc.read_h5ad(os.path.join(data_processed, "adata_integrated_with_trajectory.h5ad"))

# Filter to Walesky cells only (those with valid pseudotime)
walesky = adata[adata.obs['dataset'] == 'Walesky'].copy()

print(f"Walesky cells: {walesky.n_obs:,}")
print(f"Valid pseudotime: {walesky.obs['pseudotime'].notna().sum():,}")

# ── CRITICAL CHECK: Within-timepoint pseudotime distribution ─────────────────

print("\n" + "="*70)
print("WITHIN-TIMEPOINT PSEUDOTIME DISTRIBUTION")
print("="*70)

timepoint_order = ['0h', '6h', '24h', '48h', '96h']
walesky.obs['timepoint_str'] = pd.Categorical(
    walesky.obs['timepoint_str'], categories=timepoint_order, ordered=True
)

stats = walesky.obs.groupby('timepoint_str')['pseudotime'].describe()
print(stats.round(4))

print("\n" + "="*70)
print("WITHIN-TIMEPOINT VARIANCE (KEY METRIC)")
print("="*70)

for tp in timepoint_order:
    subset = walesky.obs[walesky.obs['timepoint_str'] == tp]['pseudotime']
    q25, q75 = subset.quantile([0.25, 0.75])
    pct_above_1 = (subset > 1.0).mean() * 100
    pct_above_3 = (subset > 3.0).mean() * 100
    print(f"\n{tp} ({len(subset):,} cells):")
    print(f"  Median: {subset.median():.4f}")
    print(f"  Std:    {subset.std():.4f}")
    print(f"  IQR:    {q25:.4f} — {q75:.4f}")
    print(f"  % cells with pseudotime > 1.0: {pct_above_1:.1f}%")
    print(f"  % cells with pseudotime > 3.0: {pct_above_3:.1f}%")

# ── VISUALIZATION ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Violin plot
sns.violinplot(
    data=walesky.obs,
    x='timepoint_str', y='pseudotime',
    order=timepoint_order,
    palette=['#2ecc71', '#e74c3c', '#e67e22', '#9b59b6', '#3498db'],
    ax=axes[0], inner='box'
)
axes[0].set_xlabel('Timepoint', fontweight='bold')
axes[0].set_ylabel('Pseudotime', fontweight='bold')
axes[0].set_title('Within-Timepoint Pseudotime Distribution', fontweight='bold')
axes[0].axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='pseudotime = 1.0')
axes[0].axhline(y=3.0, color='darkred', linestyle='--', alpha=0.5, label='pseudotime = 3.0')
axes[0].legend()

# ECDF per timepoint (shows tails clearly)
from scipy import stats as scipy_stats
colors = ['#2ecc71', '#e74c3c', '#e67e22', '#9b59b6', '#3498db']
for tp, color in zip(timepoint_order, colors):
    subset = walesky.obs[walesky.obs['timepoint_str'] == tp]['pseudotime'].dropna()
    x = np.sort(subset)
    y = np.arange(1, len(x) + 1) / len(x)
    axes[1].plot(x, y, label=f'{tp} (n={len(subset):,})', color=color, linewidth=2)

axes[1].set_xlabel('Pseudotime', fontweight='bold')
axes[1].set_ylabel('Cumulative Fraction', fontweight='bold')
axes[1].set_title('ECDF — Do Later Timepoints Have High-Pseudotime Tails?', fontweight='bold')
axes[1].axvline(x=1.0, color='red', linestyle='--', alpha=0.5)
axes[1].axvline(x=3.0, color='darkred', linestyle='--', alpha=0.5)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(
    os.path.join(base_path, "results/figures/week5_pseudotime_variance_check.png"),
    dpi=300, bbox_inches='tight'
)
plt.show()

print("\n" + "="*70)
print("INTERPRETATION GUIDE")
print("="*70)
print("What we NEED to see to proceed with TrajRisk as designed:")
print("  ✅ 0h: tight distribution, all low pseudotime")
print("  ✅ 6h: tight distribution, all high pseudotime")
print("  ✅ 24h/48h/96h: SPREAD — some cells at high pseudotime (>1.0)")
print("  ✅ 48h or 96h: tail extending toward 3.0+ (persistently stressed cells)")
print()
print("What it means if we DON'T see spread in 24h/48h/96h:")
print("  ⚠️  TrajRisk risk score would mostly = timepoint label")
print("  ⚠️  Need to reconsider the within-timepoint signal approach")
print("="*70)

In [ ]:
# Check: do the 96h high-pseudotime cells express known stress markers?
# This confirms they're biologically distinct, not trajectory noise.

high_risk_96h = walesky.obs[
    (walesky.obs['timepoint_str'] == '96h') &
    (walesky.obs['pseudotime'] > 1.0)
].index

low_risk_96h = walesky.obs[
    (walesky.obs['timepoint_str'] == '96h') &
    (walesky.obs['pseudotime'] <= 1.0)
].index

# Check Hmox1 expression in both groups
high_hmox1 = walesky[high_risk_96h, 'Hmox1'].X.mean()
low_hmox1  = walesky[low_risk_96h,  'Hmox1'].X.mean()

print(f"Hmox1 — 96h HIGH-pseudotime cells: {high_hmox1:.4f}")
print(f"Hmox1 — 96h LOW-pseudotime cells:  {low_hmox1:.4f}")
print(f"Fold difference: {high_hmox1/low_hmox1:.2f}x")

# Check Cdkn1a (p21)
high_p21 = walesky[high_risk_96h, 'Cdkn1a'].X.mean()
low_p21  = walesky[low_risk_96h,  'Cdkn1a'].X.mean()

print(f"\nCdkn1a — 96h HIGH-pseudotime cells: {high_p21:.4f}")
print(f"Cdkn1a — 96h LOW-pseudotime cells:  {low_p21:.4f}")
print(f"Fold difference: {high_p21/low_p21:.2f}x")

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os

from pathlib import Path
base_path = os.environ.get("TRAJRISK_ROOT") or str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "metadata" / "source_provenance.json").is_file()))
data_processed = os.path.join(base_path, "Data/processed")

adata   = sc.read_h5ad(os.path.join(data_processed, "adata_integrated_with_trajectory.h5ad"))
walesky = adata[adata.obs['dataset'] == 'Walesky'].copy()

# Subset to 96h APAP only — using walesky directly (already normalized)
cells_96h    = walesky.obs[walesky.obs['timepoint_str'] == '96h'].index
walesky_96h  = walesky[cells_96h].copy()

# Define groups
walesky_96h.obs['stress_group'] = np.where(
    walesky_96h.obs['pseudotime'] > 1.0,
    'High_pseudotime',
    'Low_pseudotime'
)

print(f"High-pseudotime 96h: {(walesky_96h.obs['stress_group'] == 'High_pseudotime').sum()}")
print(f"Low-pseudotime 96h:  {(walesky_96h.obs['stress_group'] == 'Low_pseudotime').sum()}")

# Differential expression
sc.tl.rank_genes_groups(
    walesky_96h,
    groupby='stress_group',
    groups=['High_pseudotime'],
    reference='Low_pseudotime',
    method='wilcoxon'
)

top_up   = sc.get.rank_genes_groups_df(walesky_96h, group='High_pseudotime').head(20)
top_down = sc.get.rank_genes_groups_df(walesky_96h, group='High_pseudotime').tail(20)

print("\nTOP 20 GENES UPREGULATED in HIGH-pseudotime 96h cells:")
print(top_up[['names', 'logfoldchanges', 'pvals_adj']].to_string())

print("\nTOP 20 GENES DOWNREGULATED in HIGH-pseudotime 96h cells:")
print(top_down[['names', 'logfoldchanges', 'pvals_adj']].to_string())

# Specific marker comparison
markers = ['Hmox1','Cdkn1a','Alb','Cyp2e1','Cyp1a2',
           'Mki67','Top2a','Pcna','Cxcl14','Tgfb1',
           'Arg1','Ass1','Gsta1','Gstm1','Atf3','Atf4','Ddit3']

present   = [m for m in markers if m in walesky_96h.var_names]
high_mask = walesky_96h.obs['stress_group'] == 'High_pseudotime'
low_mask  = walesky_96h.obs['stress_group'] == 'Low_pseudotime'

print(f"\n{'Gene':<12} {'High-pt':>12} {'Low-pt':>12} {'Ratio':>8}")
print("-" * 50)
for gene in present:
    hi = float(walesky_96h[high_mask, gene].X.mean())
    lo = float(walesky_96h[low_mask,  gene].X.mean())
    ratio = hi / lo if lo > 0 else float('nan')
    print(f"{gene:<12} {hi:>12.4f} {lo:>12.4f} {ratio:>8.2f}x")